### Data Preprocessing Pipeline

Tailored directly to **clinical_data.csv** (2,298 lesion records across 1,373 unique patients) for the 6-class skin lesion classification, clustering, and probabilistic graphical modeling project (`NEV`, `ACK`, `SEK`, `BCC`, `SCC`, `MEL`).

### Key Domain & Schema Handling for clinical_data.csv
1. **Exact Schema Alignment:** Uses exact column names (`fitspatrick`, `lesion_id`, `patient_id`, `diagnostic`, `diameter_1`, `diameter_2`, `itch`, `grew`, `hurt`, `changed`, `bleed`, `elevation`, `region`, `background_father`, `background_mother`, etc.).
2. **Structural Missingness (`804` Rows Without Full Questionnaire):** In `clinical_data.csv`, 1,494 rows have complete questionnaire/clinical measurements (`smoke`, `drink`, `pesticide`, `gender`, `skin_cancer_history`, `cancer_history`, `has_piped_water`, `has_sewage_system`, `fitspatrick`, `diameter_1`, `diameter_2`), while 804 rows have `NaN` across those fields. We explicitly capture `has_questionnaire` before imputation so models can leverage missingness patterns without leakage.
3. **Explicit `"UNK"` State Preservation:** Symptom fields (`itch`, `grew`, `hurt`, `changed`, `bleed`, `elevation`) contain `"TRUE"`, `"FALSE"`, and `"UNK"` (e.g., 402 `"UNK"` in `grew`, 396 `"UNK"` in `changed`). Rather than silently coercing `"UNK"` to `0`, we encode binary presence (`_true`) and explicit unknown indicators (`_unk`), plus `symptom_count` and `symptom_unk_count`.
4. **Leakage-Safe Splitting & Statistics:** Splits by `patient_id` (`StratifiedGroupKFold`) **before** fitting training-set medians so no test-set statistics leak into `final_train.csv` or `final_test.csv`.
5. **Dataset Variants Generated:**
   - `dataset_A_all.csv`: All 2,298 records with full feature engineering and training-median numeric imputation.
   - `dataset_B_questionnaire.csv`: Subset of 1,494 records with complete clinical questionnaire (`has_questionnaire == 1`).
   - `dataset_B_all_flag.csv`: All 2,298 records retaining `has_questionnaire` and missingness indicator flags.


In [1]:
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedGroupKFold

warnings.filterwarnings("ignore")


In [2]:
# 0. Configuration
RAW_DATA_CANDIDATES = [
    Path("../Dataset/clinical_data.csv"),
]
OUT_MODELLING_DIR = Path("../Dataset/modelling")
OUT_SPLIT_DIR = Path("../Dataset/modelling")
SEED = 42
N_SPLITS = 5

LABELS = ["NEV", "ACK", "SEK", "BCC", "SCC", "MEL"]
LABEL_TO_INT = {lbl: idx for idx, lbl in enumerate(LABELS)}
MALIGNANT_LABELS = {"BCC", "SCC", "MEL"}


## 1. Audit clinical_data.csv


In [3]:
raw_path = next((p for p in RAW_DATA_CANDIDATES if p.exists()), None)
if raw_path is None:
    raise FileNotFoundError(
        "Could not locate clinical_data.csv. Check RAW_DATA_CANDIDATES paths."
    )

df_raw = pd.read_csv(raw_path)
print(f"Loaded {raw_path} -> shape={df_raw.shape}, unique_patients={df_raw['patient_id'].nunique()}")
print("\nDiagnostic class distribution:")
print(df_raw["diagnostic"].value_counts())
print("\nMissing values per column:")
print(df_raw.isna().sum()[df_raw.isna().sum() > 0])


Loaded ..\Dataset\clinical_data.csv -> shape=(2298, 26), unique_patients=1373

Diagnostic class distribution:
diagnostic
BCC    845
ACK    730
NEV    244
SEK    235
SCC    192
MEL     52
Name: count, dtype: int64

Missing values per column:
smoke                  804
drink                  804
background_father      818
background_mother      822
pesticide              804
gender                 804
skin_cancer_history    804
cancer_history         804
has_piped_water        804
has_sewage_system      804
fitspatrick            804
diameter_1             804
diameter_2             804
dtype: int64


## 2. Clean Schema, Encode `"UNK"` / Questionnaire Flags & Engineer Domain Features


In [4]:
def clean_clinical_schema(data: pd.DataFrame) -> pd.DataFrame:
    d = data.copy()

    # 1. Target standardization (6-class diagnostic + binary is_cancer)
    d["diagnostic"] = d["diagnostic"].astype(str).str.strip().str.upper()
    d = d[d["diagnostic"].isin(LABELS)].reset_index(drop=True)
    d["y_multiclass"] = d["diagnostic"].map(LABEL_TO_INT).astype(int)
    d["is_cancer"] = d["diagnostic"].isin(MALIGNANT_LABELS).astype(int)

    # 2. Structural questionnaire availability flag (before imputing NaNs)
    q_marker_cols = [c for c in ["fitspatrick", "diameter_1", "diameter_2", "gender"] if c in d.columns]
    d["has_questionnaire"] = d[q_marker_cols].notna().any(axis=1).astype(int)

    # 3. Clean numeric columns (keep exact 'fitspatrick' spelling + alias 'fitzpatrick' if helpful)
    for col in ["age", "fitspatrick", "diameter_1", "diameter_2"]:
        if col in d.columns:
            d[col] = pd.to_numeric(d[col], errors="coerce")

    # Missingness indicators for structural blocks
    d["diameter_missing"] = (d["diameter_1"].isna() | d["diameter_2"].isna()).astype(int)
    d["fitspatrick_missing"] = d["fitspatrick"].isna().astype(int)

    # 4. Domain geometric features from lesion diameters
    d1 = d["diameter_1"].clip(lower=0)
    d2 = d["diameter_2"].clip(lower=0)
    d["mean_diameter"] = 0.5 * (d1 + d2)
    d_max = np.maximum(d1, d2)
    d_min = np.minimum(d1, d2).replace(0, np.nan)
    d["diameter_ratio"] = np.where(d["diameter_1"].notna() & d["diameter_2"].notna(), (d_max / d_min).fillna(1.0), np.nan)
    area_proxy = np.pi * (d1 / 2.0) * (d2 / 2.0)
    d["log_area"] = np.log1p(np.maximum(area_proxy, 0.0))

    # 5. Tri-state symptom encoding: TRUE -> 1, FALSE -> 0, UNK -> explicit _unk indicator
    symptom_cols = [c for c in ["itch", "grew", "hurt", "changed", "bleed", "elevation"] if c in d.columns]
    true_cols, unk_cols = [], []
    for sc in symptom_cols:
        s_clean = d[sc].fillna("UNK").astype(str).str.strip().str.upper()
        d[f"{sc}_true"] = (s_clean == "TRUE").astype(int)
        d[f"{sc}_unk"] = (s_clean == "UNK").astype(int)
        true_cols.append(f"{sc}_true")
        unk_cols.append(f"{sc}_unk")
    d = d.drop(columns=symptom_cols)

    d["symptom_count"] = d[true_cols].sum(axis=1).astype(float)
    d["symptom_unk_count"] = d[unk_cols].sum(axis=1).astype(float)
    d["age_x_symptoms"] = d["age"] * d["symptom_count"]
    d["symptom_per_decade"] = d["symptom_count"] / np.maximum(d["age"] / 10.0, 1.0)

    # 6. Questionnaire boolean columns with NaN preserved as 'MISSING' before one-hot encoding
    q_bool_cols = [
        c for c in [
            "smoke", "drink", "pesticide", "skin_cancer_history",
            "cancer_history", "has_piped_water", "has_sewage_system"
        ] if c in d.columns
    ]
    for bc in q_bool_cols:
        s_str = d[bc].where(d[bc].notna(), "MISSING").astype(str).str.strip().str.upper()
        d[bc] = s_str.replace({"TRUE": "YES", "FALSE": "NO", "1": "YES", "0": "NO"})

    # 7. Standardize ancestry/heritage typos & rare categories in background_father / background_mother
    for bg_col in [c for c in ["background_father", "background_mother"] if c in d.columns]:
        s_bg = d[bg_col].fillna("MISSING").astype(str).str.strip().str.upper()
        s_bg = s_bg.replace({"BRASIL": "BRAZIL"})
        top_cats = {"POMERANIA", "GERMANY", "ITALY", "BRAZIL", "UNK", "MISSING"}
        d[bg_col] = np.where(s_bg.isin(top_cats), s_bg, "OTHER")

    if "gender" in d.columns:
        d["gender"] = d["gender"].fillna("MISSING").astype(str).str.strip().str.upper()

    if "region" in d.columns:
        d["region"] = d["region"].fillna("MISSING").astype(str).str.strip().str.upper()

    # 8. One-hot encode categorical metadata columns
    cat_to_encode = [
        c for c in (q_bool_cols + ["gender", "region", "background_father", "background_mother"])
        if c in d.columns
    ]
    d = pd.get_dummies(d, columns=cat_to_encode, dummy_na=False, dtype=int)

    return d


df_engineered = clean_clinical_schema(df_raw)
print("Engineered dataframe shape (before train-fitted imputation):", df_engineered.shape)


Engineered dataframe shape (before train-fitted imputation): (2298, 85)


## 3. Patient-Grouped Train/Test Split & Leakage-Free Imputation
We split by `patient_id` first, compute numeric medians strictly on `train`, and apply them to both `train` and `test` (as well as full dataset variants for CV notebooks).


In [5]:
OUT_MODELLING_DIR.mkdir(parents=True, exist_ok=True)

NON_FEATURES = [
    "patient_id", "lesion_id", "img_id", "diagnostic",
    "y_multiclass", "is_cancer", "has_questionnaire", "biopsed"
]
NUMERIC_TO_IMPUTE = [
    c for c in [
        "age", "fitspatrick", "diameter_1", "diameter_2",
        "mean_diameter", "diameter_ratio", "log_area",
        "symptom_count", "symptom_unk_count", "age_x_symptoms", "symptom_per_decade"
    ] if c in df_engineered.columns
]

# Patient-grouped 80/20 split
splitter = StratifiedGroupKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
tr_idx, te_idx = next(splitter.split(df_engineered, df_engineered["y_multiclass"], df_engineered["patient_id"]))

train_df = df_engineered.iloc[tr_idx].reset_index(drop=True)
test_df = df_engineered.iloc[te_idx].reset_index(drop=True)
assert len(set(train_df["patient_id"]) & set(test_df["patient_id"])) == 0

# Fit numeric medians strictly on train_df
train_medians = {c: float(train_df[c].median()) for c in NUMERIC_TO_IMPUTE}
for c, med_val in train_medians.items():
    train_df[c] = train_df[c].fillna(med_val)
    test_df[c] = test_df[c].fillna(med_val)

# Export standard split files
feature_cols = [c for c in train_df.columns if c not in NON_FEATURES]
train_df.to_csv(OUT_SPLIT_DIR / "final_train.csv", index=False)
test_df.to_csv(OUT_SPLIT_DIR / "final_test.csv", index=False)

with open(OUT_SPLIT_DIR / "feature_columns.json", "w", encoding="utf-8") as f:
    json.dump({
        "features": feature_cols,
        "numeric_features": NUMERIC_TO_IMPUTE,
        "labels": LABELS,
        "train_medians": train_medians,
    }, f, indent=2)

# Export dataset variants for downstream CV & ablation notebooks
# Note: CV notebooks (03a/03b/03c) perform fold-wise imputation/scaling inside pipelines
dataset_a_all = df_engineered.copy()
for c, med_val in train_medians.items():
    dataset_a_all[c] = dataset_a_all[c].fillna(med_val)

dataset_b_q = dataset_a_all[dataset_a_all["has_questionnaire"] == 1].reset_index(drop=True)
dataset_b_flag = dataset_a_all.copy()

dataset_a_all.to_csv(OUT_MODELLING_DIR / "dataset_A_all.csv", index=False)
dataset_b_q.to_csv(OUT_MODELLING_DIR / "dataset_B_questionnaire.csv", index=False)
dataset_b_flag.to_csv(OUT_MODELLING_DIR / "dataset_B_all_flag.csv", index=False)

print(f"Saved final_train.csv ({len(train_df)} rows) and final_test.csv ({len(test_df)} rows)")
print(f"Saved dataset_A_all.csv ({len(dataset_a_all)} rows) & dataset_B_questionnaire.csv ({len(dataset_b_q)} rows)")
print(f"Total modelling features: {len(feature_cols)}")


Saved final_train.csv (1827 rows) and final_test.csv (471 rows)
Saved dataset_A_all.csv (2298 rows) & dataset_B_questionnaire.csv (1494 rows)
Total modelling features: 77
